## Cell 1 - Environment Setup


In [ ]:
import subprocess
import sys


def install_package(package_name):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package_name, "-q"])


install_package("ultralytics")
install_package("opencv-python-headless")
install_package("PyYAML")
install_package("matplotlib")
install_package("seaborn")
install_package("tqdm")

import os
import cv2
import yaml
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO

print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 2 - Path Configuration and Dataset Verification


In [ ]:
from pathlib import Path

LLVIP_VISIBLE_TRAIN  = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/visible/train")
LLVIP_VISIBLE_VAL    = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/visible/test")
LLVIP_INFRARED_TRAIN = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/infrared/train")
LLVIP_INFRARED_VAL   = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/infrared/test")
LLVIP_ANNOTATIONS    = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/Annotations")

PROJECT_ROOT         = Path("/kaggle/working/llvip_fusion_yolov8")
FUSED_TRAIN          = PROJECT_ROOT / "images" / "train"
FUSED_VAL            = PROJECT_ROOT / "images" / "val"
DATASET_LABELS_TRAIN = PROJECT_ROOT / "labels" / "train"
DATASET_LABELS_VAL   = PROJECT_ROOT / "labels" / "val"
YOLO_RUNS_DIR        = PROJECT_ROOT / "runs"

for directory in [FUSED_TRAIN, FUSED_VAL, DATASET_LABELS_TRAIN, DATASET_LABELS_VAL, YOLO_RUNS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

VISIBLE_TRAIN_COUNT  = len(list(LLVIP_VISIBLE_TRAIN.glob("*.jpg")))  if LLVIP_VISIBLE_TRAIN.exists() else 0
INFRARED_TRAIN_COUNT = len(list(LLVIP_INFRARED_TRAIN.glob("*.jpg"))) if LLVIP_INFRARED_TRAIN.exists() else 0
VISIBLE_VAL_COUNT    = len(list(LLVIP_VISIBLE_VAL.glob("*.jpg")))    if LLVIP_VISIBLE_VAL.exists() else 0
ANNOTATIONS_COUNT    = len(list(LLVIP_ANNOTATIONS.glob("*.xml")))    if LLVIP_ANNOTATIONS.exists() else 0

print("=" * 55)
print("        LLVIP DATASET STRUCTURE VERIFICATION")
print("=" * 55)
print(f"  Visible  train images : {VISIBLE_TRAIN_COUNT:>6,}")
print(f"  Infrared train images : {INFRARED_TRAIN_COUNT:>6,}")
print(f"  Visible  val   images : {VISIBLE_VAL_COUNT:>6,}")
print(f"  Annotations (.xml)    : {ANNOTATIONS_COUNT:>6,}")
print("=" * 55)

DATASET_READY = all([
    VISIBLE_TRAIN_COUNT > 0,
    INFRARED_TRAIN_COUNT > 0,
    ANNOTATIONS_COUNT > 0,
])

if DATASET_READY:
    print("  STATUS: Dataset loaded successfully.")
else:
    print("  WARNING: One or more directories are empty or missing.")
    for name, path in [
        ("visible/train", LLVIP_VISIBLE_TRAIN),
        ("infrared/train", LLVIP_INFRARED_TRAIN),
        ("Annotations", LLVIP_ANNOTATIONS),
    ]:
        status = "OK" if path.exists() else "MISSING"
        print(f"    [{status}] {path}")

print("=" * 55)

## Cell 3 - Image-Level Fusion Pipeline


In [ ]:
FUSION_ALPHA = 0.5
FUSION_BETA  = 0.5


def load_visible_image(visible_path: Path) -> np.ndarray:
    image = cv2.imread(str(visible_path))
    if image is None:
        raise FileNotFoundError(f"Cannot load visible image: {visible_path}")
    return image


def load_infrared_as_bgr(infrared_path: Path) -> np.ndarray:
    image_gray = cv2.imread(str(infrared_path), cv2.IMREAD_GRAYSCALE)
    if image_gray is None:
        raise FileNotFoundError(f"Cannot load infrared image: {infrared_path}")
    image_bgr = cv2.cvtColor(image_gray, cv2.COLOR_GRAY2BGR)
    return image_bgr


def align_image_sizes(visible_bgr: np.ndarray, infrared_bgr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    target_height, target_width = visible_bgr.shape[:2]
    if infrared_bgr.shape[:2] != (target_height, target_width):
        infrared_bgr = cv2.resize(infrared_bgr, (target_width, target_height), interpolation=cv2.INTER_LINEAR)
    return visible_bgr, infrared_bgr


def fuse_image_pair(visible_bgr: np.ndarray, infrared_bgr: np.ndarray) -> np.ndarray:
    visible_bgr, infrared_bgr = align_image_sizes(visible_bgr, infrared_bgr)
    fused = cv2.addWeighted(visible_bgr, FUSION_ALPHA, infrared_bgr, FUSION_BETA, gamma=0)
    return fused


def process_single_pair(
    visible_path: Path,
    infrared_path: Path,
    output_path: Path,
) -> bool:
    try:
        visible_bgr  = load_visible_image(visible_path)
        infrared_bgr = load_infrared_as_bgr(infrared_path)
        fused_image  = fuse_image_pair(visible_bgr, infrared_bgr)
        cv2.imwrite(str(output_path), fused_image)
        return True
    except (FileNotFoundError, cv2.error):
        return False


def run_fusion_pipeline(
    visible_dir: Path,
    infrared_dir: Path,
    output_dir: Path,
    split_name: str,
) -> int:
    visible_images = sorted(visible_dir.glob("*.jpg"))
    if not visible_images:
        visible_images = sorted(visible_dir.glob("*.png"))

    success_count = 0
    failed_stems   = []

    progress_bar = tqdm(visible_images, desc=f"Fusing [{split_name}]", unit="pair")
    for visible_path in progress_bar:
        stem = visible_path.stem
        infrared_path = infrared_dir / visible_path.name
        output_path   = output_dir / f"{stem}.jpg"

        if output_path.exists():
            success_count += 1
            continue

        if not infrared_path.exists():
            infrared_candidates = list(infrared_dir.glob(f"{stem}.*"))
            if not infrared_candidates:
                failed_stems.append(stem)
                continue
            infrared_path = infrared_candidates[0]

        fused_ok = process_single_pair(visible_path, infrared_path, output_path)
        if fused_ok:
            success_count += 1
        else:
            failed_stems.append(stem)

        progress_bar.set_postfix({"success": success_count, "failed": len(failed_stems)})

    print(f"\n  [{split_name}] Fused: {success_count:,} | Failed: {len(failed_stems):,}")
    if failed_stems:
        print(f"  Failed stems (first 5): {failed_stems[:5]}")

    return success_count


print("Starting image-level fusion pipeline...")
print(f"  Alpha (Visible)  : {FUSION_ALPHA}")
print(f"  Beta  (Infrared) : {FUSION_BETA}")
print("-" * 55)

train_fused_count = run_fusion_pipeline(
    visible_dir  = LLVIP_VISIBLE_TRAIN,
    infrared_dir = LLVIP_INFRARED_TRAIN,
    output_dir   = FUSED_TRAIN,
    split_name   = "train",
)

val_fused_count = run_fusion_pipeline(
    visible_dir  = LLVIP_VISIBLE_VAL,
    infrared_dir = LLVIP_INFRARED_VAL,
    output_dir   = FUSED_VAL,
    split_name   = "val",
)

print("-" * 55)
print(f"  Total fused images: {train_fused_count + val_fused_count:,}")
print("  Fusion pipeline complete.")


## Cell 4 - XML to YOLO Conversion and data.yaml


In [ ]:
import xml.etree.ElementTree as ET

TRAIN_RATIO = 0.85


def parse_voc_xml_to_yolo(xml_path: Path, image_width: int, image_height: int) -> list[str]:
    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    yolo_lines = []

    for obj in root.findall("object"):
        class_name = obj.find("name").text.strip().lower()
        if class_name not in ("person", "pedestrian"):
            continue

        bndbox  = obj.find("bndbox")
        xmin    = float(bndbox.find("xmin").text)
        ymin    = float(bndbox.find("ymin").text)
        xmax    = float(bndbox.find("xmax").text)
        ymax    = float(bndbox.find("ymax").text)

        xmin = max(0.0, min(xmin, image_width))
        ymin = max(0.0, min(ymin, image_height))
        xmax = max(0.0, min(xmax, image_width))
        ymax = max(0.0, min(ymax, image_height))

        cx = (xmin + xmax) / 2.0 / image_width
        cy = (ymin + ymax) / 2.0 / image_height
        bw = (xmax - xmin) / image_width
        bh = (ymax - ymin) / image_height

        if bw <= 0 or bh <= 0:
            continue

        yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    return yolo_lines


def get_image_dimensions(image_path: Path) -> tuple[int, int]:
    img = cv2.imread(str(image_path))
    if img is None:
        return 640, 512
    h, w = img.shape[:2]
    return w, h


def convert_annotations_for_split(
    fused_dir: Path,
    annotation_dir: Path,
    label_output_dir: Path,
    split_name: str,
) -> int:
    fused_images  = sorted(fused_dir.glob("*.jpg"))
    converted     = 0
    missing_xml   = 0

    for image_path in tqdm(fused_images, desc=f"Converting XML->YOLO [{split_name}]", unit="file"):
        stem       = image_path.stem
        xml_path   = annotation_dir / f"{stem}.xml"
        label_path = label_output_dir / f"{stem}.txt"

        if label_path.exists():
            converted += 1
            continue

        if not xml_path.exists():
            missing_xml += 1
            open(str(label_path), "w").close()
            continue

        img_w, img_h = get_image_dimensions(image_path)
        yolo_lines   = parse_voc_xml_to_yolo(xml_path, img_w, img_h)

        with open(str(label_path), "w") as f:
            f.write("\n".join(yolo_lines))

        converted += 1

    print(f"  [{split_name}] Converted: {converted:,} | Missing XML: {missing_xml:,}")
    return converted


def create_yolo_data_yaml(output_path: Path) -> None:
    yaml_content = {
        "path"  : str(PROJECT_ROOT),
        "train" : "images/train",
        "val"   : "images/val",
        "nc"    : 1,
        "names" : ["pedestrian"],
    }
    with open(output_path, "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
    print(f"  data.yaml written: {output_path}")


def verify_image_label_alignment(image_dir: Path, label_dir: Path, split_name: str) -> None:
    image_stems = {p.stem for p in image_dir.glob("*.jpg")}
    label_stems = {p.stem for p in label_dir.glob("*.txt")}
    matched     = image_stems & label_stems
    unmatched   = image_stems - label_stems
    print(f"  [{split_name}] Matched: {len(matched):,} | Unmatched: {len(unmatched):,}")


def plot_label_distribution(label_dir: Path, split_name: str, max_samples: int = 500) -> None:
    all_boxes    = []
    label_files  = list(label_dir.glob("*.txt"))
    sample_files = random.sample(label_files, min(max_samples, len(label_files)))

    for label_path in sample_files:
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    all_boxes.append([float(x) for x in parts[1:]])

    if not all_boxes:
        print(f"  No boxes found in [{split_name}].")
        return

    boxes   = np.array(all_boxes)
    widths  = boxes[:, 2]
    heights = boxes[:, 3]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Bounding Box Distribution - {split_name.capitalize()} Split", fontsize=13, fontweight="bold")

    axes[0].hist(widths,  bins=40, color="#4C72B0", edgecolor="white")
    axes[0].set_title("Normalized Box Width")
    axes[0].set_xlabel("Width (0-1)")
    axes[0].set_ylabel("Count")
    axes[0].axvline(widths.mean(), color="red", linestyle="--", label=f"Mean={widths.mean():.3f}")
    axes[0].legend()

    axes[1].hist(heights, bins=40, color="#DD8452", edgecolor="white")
    axes[1].set_title("Normalized Box Height")
    axes[1].set_xlabel("Height (0-1)")
    axes[1].set_ylabel("Count")
    axes[1].axvline(heights.mean(), color="red", linestyle="--", label=f"Mean={heights.mean():.3f}")
    axes[1].legend()

    plt.tight_layout()
    out = PROJECT_ROOT / f"label_distribution_{split_name}.png"
    plt.savefig(str(out), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Total boxes in sample: {len(all_boxes):,}")


print("Converting Pascal VOC XML annotations to YOLO format...")
print("-" * 55)

train_converted = convert_annotations_for_split(
    fused_dir          = FUSED_TRAIN,
    annotation_dir     = LLVIP_ANNOTATIONS,
    label_output_dir   = DATASET_LABELS_TRAIN,
    split_name         = "train",
)

val_converted = convert_annotations_for_split(
    fused_dir          = FUSED_VAL,
    annotation_dir     = LLVIP_ANNOTATIONS,
    label_output_dir   = DATASET_LABELS_VAL,
    split_name         = "val",
)

verify_image_label_alignment(FUSED_TRAIN, DATASET_LABELS_TRAIN, "train")
verify_image_label_alignment(FUSED_VAL,   DATASET_LABELS_VAL,   "val")

DATA_YAML_PATH = PROJECT_ROOT / "data.yaml"
create_yolo_data_yaml(DATA_YAML_PATH)

print("\ndata.yaml content:")
with open(DATA_YAML_PATH, "r") as f:
    print(f.read())

print("Generating label distribution plot...")
plot_label_distribution(DATASET_LABELS_TRAIN, "train")


## Cell 5 - Qualitative Fusion Visualization


In [ ]:
def draw_yolo_boxes_on_image(
    image_bgr: np.ndarray,
    label_path: Path,
    color: tuple = (0, 255, 0),
    thickness: int = 2,
) -> np.ndarray:
    annotated = image_bgr.copy()
    h, w = annotated.shape[:2]

    if not label_path.exists():
        return annotated

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, cx, cy, bw, bh = [float(x) for x in parts]
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, thickness)

    return annotated


def visualize_fusion_samples(
    visible_dir: Path,
    infrared_dir: Path,
    fused_dir: Path,
    label_dir: Path,
    num_samples: int = 4,
) -> None:
    fused_images = sorted(fused_dir.glob("*.jpg"))
    sample_paths = random.sample(fused_images, min(num_samples, len(fused_images)))

    fig, axes = plt.subplots(num_samples, 3, figsize=(18, num_samples * 4))
    if num_samples == 1:
        axes = [axes]

    column_titles = ["Visible (RGB)", "Infrared (Thermal)", "Fused (50% + 50%) + GT Boxes"]

    for col_idx, title in enumerate(column_titles):
        axes[0][col_idx].set_title(title, fontsize=13, fontweight="bold", pad=10)

    for row_idx, fused_path in enumerate(sample_paths):
        stem          = fused_path.stem
        visible_path  = visible_dir  / fused_path.name
        infrared_path = infrared_dir / fused_path.name
        label_path    = label_dir    / f"{stem}.txt"

        visible_bgr  = cv2.imread(str(visible_path))  if visible_path.exists()  else np.zeros((640, 640, 3), dtype=np.uint8)
        infrared_bgr = cv2.imread(str(infrared_path)) if infrared_path.exists() else np.zeros((640, 640, 3), dtype=np.uint8)
        fused_bgr    = cv2.imread(str(fused_path))

        fused_annotated = draw_yolo_boxes_on_image(fused_bgr, label_path, color=(0, 255, 0))

        axes[row_idx][0].imshow(cv2.cvtColor(visible_bgr,     cv2.COLOR_BGR2RGB))
        axes[row_idx][1].imshow(cv2.cvtColor(infrared_bgr,    cv2.COLOR_BGR2RGB))
        axes[row_idx][2].imshow(cv2.cvtColor(fused_annotated, cv2.COLOR_BGR2RGB))

        for col_idx in range(3):
            axes[row_idx][col_idx].axis("off")
            axes[row_idx][col_idx].set_aspect("equal")

        axes[row_idx][0].set_ylabel(f"Sample {row_idx + 1}\n({stem})", fontsize=9, rotation=0, labelpad=80, va="center")

    plt.suptitle(
        "LLVIP Image-Level Fusion: Visible vs Infrared vs Fused",
        fontsize=15,
        fontweight="bold",
        y=1.01,
    )
    plt.tight_layout()
    output_path = PROJECT_ROOT / "fusion_samples_visualization.png"
    plt.savefig(str(output_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved fusion visualization to: {output_path}")


print("Generating qualitative fusion visualization...")
random.seed(42)
visualize_fusion_samples(
    visible_dir  = LLVIP_VISIBLE_TRAIN,
    infrared_dir = LLVIP_INFRARED_TRAIN,
    fused_dir    = FUSED_TRAIN,
    label_dir    = DATASET_LABELS_TRAIN,
    num_samples  = 4,
)


## Cell 6 - YOLOv8s Fine-Tuning


In [ ]:
YOLO_MODEL_VARIANT = "yolov8s.pt"
TRAINING_EPOCHS    = 50
IMAGE_SIZE         = 640
BATCH_SIZE         = 16
LEARNING_RATE_0    = 0.01
LEARNING_RATE_F    = 0.001
WEIGHT_DECAY       = 5e-4
WARMUP_EPOCHS      = 3
PATIENCE           = 15
WORKERS            = 4
EXPERIMENT_NAME    = "llvip_fusion_yolov8s"


def build_training_config() -> dict:
    return {
        "data"        : str(DATA_YAML_PATH),
        "epochs"      : TRAINING_EPOCHS,
        "imgsz"       : IMAGE_SIZE,
        "batch"       : BATCH_SIZE,
        "lr0"         : LEARNING_RATE_0,
        "lrf"         : LEARNING_RATE_F,
        "weight_decay": WEIGHT_DECAY,
        "warmup_epochs": WARMUP_EPOCHS,
        "patience"    : PATIENCE,
        "workers"     : WORKERS,
        "device"      : 0 if DEVICE == "cuda" else "cpu",
        "project"     : str(YOLO_RUNS_DIR),
        "name"        : EXPERIMENT_NAME,
        "exist_ok"    : True,
        "pretrained"  : True,
        "optimizer"   : "AdamW",
        "cos_lr"      : True,
        "mosaic"      : 1.0,
        "mixup"       : 0.1,
        "copy_paste"  : 0.1,
        "hsv_h"       : 0.015,
        "hsv_s"       : 0.5,
        "hsv_v"       : 0.4,
        "degrees"     : 5.0,
        "translate"   : 0.1,
        "scale"       : 0.5,
        "flipud"      : 0.0,
        "fliplr"      : 0.5,
        "verbose"     : True,
        "seed"        : 42,
        "amp"         : True,
        "plots"       : True,
    }


print("=" * 55)
print("          YOLOV8 FINE-TUNING CONFIGURATION")
print("=" * 55)
print(f"  Model variant   : {YOLO_MODEL_VARIANT}")
print(f"  Training epochs : {TRAINING_EPOCHS}")
print(f"  Image size      : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Optimizer       : AdamW (cosine LR)")
print(f"  Learning rate   : {LEARNING_RATE_0} → {LEARNING_RATE_F}")
print(f"  Warmup epochs   : {WARMUP_EPOCHS}")
print(f"  Early stopping  : patience={PATIENCE}")
print(f"  Device          : {DEVICE.upper()}")
print(f"  Experiment name : {EXPERIMENT_NAME}")
print("=" * 55)

yolo_model   = YOLO(YOLO_MODEL_VARIANT)
training_cfg = build_training_config()

print("\nStarting training...")
training_results = yolo_model.train(**training_cfg)

BEST_MODEL_PATH  = YOLO_RUNS_DIR / EXPERIMENT_NAME / "weights" / "best.pt"
LAST_MODEL_PATH  = YOLO_RUNS_DIR / EXPERIMENT_NAME / "weights" / "last.pt"

print("\n" + "=" * 55)
print("  Training complete.")
print(f"  Best weights : {BEST_MODEL_PATH}")
print(f"  Last weights : {LAST_MODEL_PATH}")
print("=" * 55)






# ── Cell lưu model ra output để save vĩnh viễn ──────────────
import shutil

SAVE_DIR = "/kaggle/working"   # đây là output được Kaggle lưu lại

# Copy best.pt và last.pt ra thẳng /kaggle/working
src = "/kaggle/working/llvip_fusion_yolov8/runs/llvip_fusion_yolov8s/weights/"
shutil.copy(src + "best.pt", SAVE_DIR + "/best.pt")
shutil.copy(src + "last.pt", SAVE_DIR + "/last.pt")

print("✅ Đã copy model ra output:")
print(f"   {SAVE_DIR}/best.pt")
print(f"   {SAVE_DIR}/last.pt")

## Cell 7 - Model Evaluation and Metrics


In [ ]:
def run_validation(model_path: Path, data_yaml_path: Path) -> dict:
    evaluation_model = YOLO(str(model_path))
    metrics = evaluation_model.val(
        data    = str(data_yaml_path),
        imgsz   = IMAGE_SIZE,
        batch   = BATCH_SIZE,
        device  = 0 if DEVICE == "cuda" else "cpu",
        workers = WORKERS,
        verbose = True,
    )
    return metrics


def extract_metric_values(metrics) -> dict:
    return {
        "Precision"    : float(metrics.box.mp),
        "Recall"       : float(metrics.box.mr),
        "mAP@50"       : float(metrics.box.map50),
        "mAP@50-95"    : float(metrics.box.map),
        "F1-Score"     : 2 * float(metrics.box.mp) * float(metrics.box.mr)
                         / (float(metrics.box.mp) + float(metrics.box.mr) + 1e-9),
    }


def print_metrics_table(metric_values: dict, model_label: str) -> None:
    bar = "=" * 45
    print(f"\n{bar}")
    print(f"   EVALUATION RESULTS — {model_label}")
    print(bar)
    for metric_name, value in metric_values.items():
        status = "✓" if value >= 0.5 else "△"
        print(f"   {status}  {metric_name:<18}: {value:.4f}  ({value * 100:.2f}%)")
    print(bar)


def plot_metrics_bar_chart(metric_values: dict, model_label: str, output_path: Path) -> None:
    metric_names  = list(metric_values.keys())
    metric_scores = list(metric_values.values())
    colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(metric_names, metric_scores, color=colors[:len(metric_names)], edgecolor="white", linewidth=1.2, width=0.55)

    for bar, score in zip(bars, metric_scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{score:.4f}",
            ha="center", va="bottom", fontsize=11, fontweight="bold",
        )

    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"Detection Performance Metrics — {model_label}", fontsize=13, fontweight="bold", pad=15)
    ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="0.5 threshold")
    ax.axhline(y=1.0, color="black", linestyle="-",  linewidth=0.8, alpha=0.3)
    ax.legend(fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(str(output_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved metrics bar chart to: {output_path}")


def load_training_csv(runs_dir: Path, experiment_name: str) -> dict:
    csv_path = runs_dir / experiment_name / "results.csv"
    if not csv_path.exists():
        print(f"  WARNING: results.csv not found at {csv_path}")
        return {}

    import csv
    rows = []
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append({k.strip(): float(v.strip()) for k, v in row.items() if v.strip()})

    return rows


def plot_training_loss_curves(training_rows: list, output_path: Path) -> None:
    if not training_rows:
        print("  No training data available for loss curve plotting.")
        return

    epochs          = [row["epoch"]                for row in training_rows]
    train_box_loss  = [row.get("train/box_loss", 0) for row in training_rows]
    train_cls_loss  = [row.get("train/cls_loss", 0) for row in training_rows]
    train_dfl_loss  = [row.get("train/dfl_loss", 0) for row in training_rows]
    val_box_loss    = [row.get("val/box_loss", 0)   for row in training_rows]
    val_cls_loss    = [row.get("val/cls_loss", 0)   for row in training_rows]
    map50_curve     = [row.get("metrics/mAP50(B)", 0)    for row in training_rows]
    map50_95_curve  = [row.get("metrics/mAP50-95(B)", 0) for row in training_rows]
    precision_curve = [row.get("metrics/precision(B)", 0) for row in training_rows]
    recall_curve    = [row.get("metrics/recall(B)", 0)    for row in training_rows]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("YOLOv8 Training Curves — LLVIP Fused Dataset", fontsize=14, fontweight="bold", y=1.01)

    def plot_curve(ax, x, y_train, y_val, title, ylabel, color_train, color_val):
        ax.plot(x, y_train, color=color_train, linewidth=2, label="Train")
        if any(v != 0 for v in y_val):
            ax.plot(x, y_val, color=color_val, linewidth=2, linestyle="--", label="Val")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="y", alpha=0.3)

    plot_curve(axes[0][0], epochs, train_box_loss, val_box_loss,   "Box Loss",  "Loss", "#E15759", "#E15759")
    plot_curve(axes[0][1], epochs, train_cls_loss, val_cls_loss,   "Cls Loss",  "Loss", "#F28E2B", "#F28E2B")
    plot_curve(axes[0][2], epochs, train_dfl_loss, [0]*len(epochs), "DFL Loss", "Loss", "#76B7B2", "#76B7B2")

    axes[1][0].plot(epochs, map50_curve,    color="#4E79A7", linewidth=2, label="mAP@50")
    axes[1][0].plot(epochs, map50_95_curve, color="#59A14F", linewidth=2, linestyle="--", label="mAP@50-95")
    axes[1][0].set_title("mAP Curves", fontsize=11, fontweight="bold")
    axes[1][0].set_xlabel("Epoch")
    axes[1][0].set_ylabel("mAP")
    axes[1][0].legend()
    axes[1][0].spines["top"].set_visible(False)
    axes[1][0].spines["right"].set_visible(False)
    axes[1][0].grid(axis="y", alpha=0.3)

    axes[1][1].plot(epochs, precision_curve, color="#B07AA1", linewidth=2, label="Precision")
    axes[1][1].plot(epochs, recall_curve,    color="#FF9DA7", linewidth=2, linestyle="--", label="Recall")
    axes[1][1].set_title("Precision & Recall", fontsize=11, fontweight="bold")
    axes[1][1].set_xlabel("Epoch")
    axes[1][1].set_ylabel("Score")
    axes[1][1].legend()
    axes[1][1].spines["top"].set_visible(False)
    axes[1][1].spines["right"].set_visible(False)
    axes[1][1].grid(axis="y", alpha=0.3)

    f1_curve = [
        2 * p * r / (p + r + 1e-9)
        for p, r in zip(precision_curve, recall_curve)
    ]
    axes[1][2].plot(epochs, f1_curve, color="#EDC948", linewidth=2, label="F1-Score")
    axes[1][2].set_title("F1-Score Curve", fontsize=11, fontweight="bold")
    axes[1][2].set_xlabel("Epoch")
    axes[1][2].set_ylabel("F1")
    axes[1][2].legend()
    axes[1][2].spines["top"].set_visible(False)
    axes[1][2].spines["right"].set_visible(False)
    axes[1][2].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(output_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved training curves to: {output_path}")


print("Running evaluation on best checkpoint...")
print("-" * 55)

validation_metrics = run_validation(BEST_MODEL_PATH, DATA_YAML_PATH)
metric_values      = extract_metric_values(validation_metrics)

print_metrics_table(metric_values, model_label=f"YOLOv8s — LLVIP Fusion (best.pt)")

plot_metrics_bar_chart(
    metric_values,
    model_label = "YOLOv8s — LLVIP Fusion",
    output_path = PROJECT_ROOT / "metrics_bar_chart.png",
)

training_rows = load_training_csv(YOLO_RUNS_DIR, EXPERIMENT_NAME)
plot_training_loss_curves(
    training_rows,
    output_path = PROJECT_ROOT / "training_curves.png",
)


## Cell 8 - Inference Visualization


In [ ]:
INFERENCE_CONFIDENCE_THRESHOLD = 0.25
INFERENCE_IOU_THRESHOLD        = 0.45
INFERENCE_NUM_SAMPLES          = 8


def run_inference_on_image(
    model: YOLO,
    image_path: Path,
    conf_threshold: float,
    iou_threshold: float,
) -> np.ndarray:
    results = model.predict(
        source = str(image_path),
        conf   = conf_threshold,
        iou    = iou_threshold,
        imgsz  = IMAGE_SIZE,
        device = 0 if DEVICE == "cuda" else "cpu",
        verbose = False,
    )
    annotated_bgr = results[0].plot()
    return annotated_bgr


def count_detections(model: YOLO, image_path: Path, conf_threshold: float) -> int:
    results = model.predict(
        source  = str(image_path),
        conf    = conf_threshold,
        imgsz   = IMAGE_SIZE,
        device  = 0 if DEVICE == "cuda" else "cpu",
        verbose = False,
    )
    return len(results[0].boxes)


def visualize_inference_samples(
    model: YOLO,
    fused_val_dir: Path,
    label_val_dir: Path,
    num_samples: int,
    conf_threshold: float,
    iou_threshold: float,
    output_path: Path,
) -> None:
    val_images   = sorted(fused_val_dir.glob("*.jpg"))
    sample_paths = random.sample(val_images, min(num_samples, len(val_images)))

    fig, axes = plt.subplots(num_samples, 2, figsize=(16, num_samples * 4))
    if num_samples == 1:
        axes = [axes]

    axes[0][0].set_title("Ground Truth Labels", fontsize=13, fontweight="bold", pad=10)
    axes[0][1].set_title(f"YOLOv8s Predictions (conf≥{conf_threshold})", fontsize=13, fontweight="bold", pad=10)

    for row_idx, image_path in enumerate(sample_paths):
        stem       = image_path.stem
        label_path = label_val_dir / f"{stem}.txt"

        fused_bgr       = cv2.imread(str(image_path))
        gt_annotated    = draw_yolo_boxes_on_image(fused_bgr, label_path, color=(0, 200, 0), thickness=2)
        pred_annotated  = run_inference_on_image(model, image_path, conf_threshold, iou_threshold)

        gt_box_count   = sum(1 for _ in open(label_path) if _.strip()) if label_path.exists() else 0
        pred_box_count = count_detections(model, image_path, conf_threshold)

        axes[row_idx][0].imshow(cv2.cvtColor(gt_annotated,   cv2.COLOR_BGR2RGB))
        axes[row_idx][1].imshow(cv2.cvtColor(pred_annotated, cv2.COLOR_BGR2RGB))

        axes[row_idx][0].set_title(f"GT: {gt_box_count} pedestrian(s)", fontsize=10, pad=6)
        axes[row_idx][1].set_title(f"Pred: {pred_box_count} pedestrian(s)", fontsize=10, pad=6)

        for col_idx in range(2):
            axes[row_idx][col_idx].axis("off")

    plt.suptitle(
        "Qualitative Inference Results — LLVIP Validation Set",
        fontsize=14, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    plt.savefig(str(output_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved inference visualization to: {output_path}")


def plot_confidence_distribution(
    model: YOLO,
    fused_val_dir: Path,
    max_samples: int,
    output_path: Path,
) -> None:
    val_images     = sorted(fused_val_dir.glob("*.jpg"))
    sample_paths   = random.sample(val_images, min(max_samples, len(val_images)))
    all_confidences = []

    for image_path in tqdm(sample_paths, desc="Collecting confidences", unit="img"):
        results = model.predict(
            source  = str(image_path),
            conf    = 0.01,
            imgsz   = IMAGE_SIZE,
            device  = 0 if DEVICE == "cuda" else "cpu",
            verbose = False,
        )
        confs = results[0].boxes.conf.cpu().numpy().tolist() if len(results[0].boxes) else []
        all_confidences.extend(confs)

    if not all_confidences:
        print("  No detections found for confidence distribution plot.")
        return

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(all_confidences, bins=50, color="#4C72B0", edgecolor="white", linewidth=0.5)
    ax.axvline(x=INFERENCE_CONFIDENCE_THRESHOLD, color="red", linestyle="--", linewidth=1.5, label=f"Threshold={INFERENCE_CONFIDENCE_THRESHOLD}")
    ax.set_title("Prediction Confidence Score Distribution", fontsize=13, fontweight="bold")
    ax.set_xlabel("Confidence Score")
    ax.set_ylabel("Count")
    ax.legend(fontsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(str(output_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Total detections collected: {len(all_confidences):,}")
    print(f"  Mean confidence: {np.mean(all_confidences):.4f}")
    print(f"  Saved confidence distribution to: {output_path}")


print("Loading best model for inference visualization...")
inference_model = YOLO(str(BEST_MODEL_PATH))
print(f"  Model loaded: {BEST_MODEL_PATH.name}")

random.seed(0)
visualize_inference_samples(
    model          = inference_model,
    fused_val_dir  = FUSED_VAL,
    label_val_dir  = DATASET_LABELS_VAL,
    num_samples    = INFERENCE_NUM_SAMPLES,
    conf_threshold = INFERENCE_CONFIDENCE_THRESHOLD,
    iou_threshold  = INFERENCE_IOU_THRESHOLD,
    output_path    = PROJECT_ROOT / "inference_samples.png",
)

plot_confidence_distribution(
    model         = inference_model,
    fused_val_dir = FUSED_VAL,
    max_samples   = 200,
    output_path   = PROJECT_ROOT / "confidence_distribution.png",
)


## Cell 9 - Final Summary Report


In [ ]:
def collect_output_artifacts(project_root: Path) -> list[Path]:
    artifact_patterns = ["*.png", "*.yaml", "*.csv"]
    artifacts = []
    for pattern in artifact_patterns:
        artifacts.extend(sorted(project_root.glob(pattern)))

    weights_dir = YOLO_RUNS_DIR / EXPERIMENT_NAME / "weights"
    if weights_dir.exists():
        artifacts.extend(sorted(weights_dir.glob("*.pt")))

    return artifacts


def print_final_report(
    metric_values: dict,
    train_fused_count: int,
    val_fused_count: int,
    fusion_alpha: float,
    fusion_beta: float,
    model_variant: str,
    experiment_name: str,
    best_model_path: Path,
) -> None:
    separator = "=" * 60

    print(separator)
    print("         FINAL PROJECT SUMMARY REPORT")
    print("         Low-Light Pedestrian Detection")
    print("         LLVIP Dataset + YOLOv8 Fusion")
    print(separator)

    print("\n  [DATASET]")
    print(f"    Dataset        : LLVIP (Low-Light Visible-Infrared Paired)")
    print(f"    Train images   : {train_fused_count:,}")
    print(f"    Val   images   : {val_fused_count:,}")
    print(f"    Total pairs    : {train_fused_count + val_fused_count:,}")
    print(f"    Classes        : 1 (pedestrian)")

    print("\n  [METHODOLOGY]")
    print(f"    Fusion type    : Image-level (cv2.addWeighted)")
    print(f"    Alpha (RGB)    : {fusion_alpha}")
    print(f"    Beta  (IR)     : {fusion_beta}")
    print(f"    Rationale      : Visible provides texture/context;")
    print(f"                     IR highlights body heat in darkness.")

    print("\n  [MODEL]")
    print(f"    Architecture   : {model_variant}")
    print(f"    Strategy       : Fine-tuning from COCO pre-trained weights")
    print(f"    Optimizer      : AdamW with cosine LR schedule")
    print(f"    Best weights   : {best_model_path.name}")

    print("\n  [PERFORMANCE METRICS — Validation Set]")
    for metric_name, value in metric_values.items():
        bar_length = int(value * 30)
        bar_str    = "█" * bar_length + "░" * (30 - bar_length)
        print(f"    {metric_name:<18}: {value:.4f}  |{bar_str}|")

    print("\n  [KEY FINDINGS]")
    map50    = metric_values.get("mAP@50", 0)
    map5095  = metric_values.get("mAP@50-95", 0)
    recall   = metric_values.get("Recall", 0)
    prec     = metric_values.get("Precision", 0)

    if map50 >= 0.85:
        finding_map50 = f"Excellent mAP@50={map50:.3f} — model reliably detects pedestrians at night."
    elif map50 >= 0.70:
        finding_map50 = f"Good mAP@50={map50:.3f} — fusion significantly improves over visible-only baseline."
    else:
        finding_map50 = f"mAP@50={map50:.3f} — further tuning or deeper model recommended."

    if recall >= 0.85:
        finding_recall = f"High Recall={recall:.3f} — few missed pedestrian detections."
    else:
        finding_recall = f"Recall={recall:.3f} — some missed detections; consider lower conf threshold."

    print(f"    • {finding_map50}")
    print(f"    • {finding_recall}")
    print(f"    • Image-level fusion enabled a single standard YOLOv8")
    print(f"      backbone to leverage both RGB texture and IR thermal")
    print(f"      signal without architectural modification.")
    print(f"    • mAP@50-95={map5095:.3f} reflects localization quality")
    print(f"      across multiple IoU thresholds.")

    print("\n  [OUTPUT ARTIFACTS]")
    artifacts = collect_output_artifacts(PROJECT_ROOT)
    for artifact in artifacts:
        size_kb = artifact.stat().st_size / 1024
        print(f"    • {artifact.name:<45} ({size_kb:>7.1f} KB)")

    print("\n" + separator)
    print("  STATUS: Project complete. Ready for submission.")
    print(separator)


print_final_report(
    metric_values      = metric_values,
    train_fused_count  = train_fused_count,
    val_fused_count    = val_fused_count,
    fusion_alpha       = FUSION_ALPHA,
    fusion_beta        = FUSION_BETA,
    model_variant      = YOLO_MODEL_VARIANT,
    experiment_name    = EXPERIMENT_NAME,
    best_model_path    = BEST_MODEL_PATH,
)


# Gradio Demo — LLVIP Pedestrian Detection
**Chạy cell dưới đây để khởi động giao diện demo tương tác.**

> Yêu cầu: đã chạy xong pipeline training (Cell 1–9) để có `best.pt`


In [ ]:
# ============================================================
# CELL GUI - GRADIO DEMO: Low-Light Pedestrian Detection
# LLVIP Visible+Infrared Fusion + YOLOv8s
# ============================================================

import subprocess, sys

# Cài đặt các thư viện cần thiết
subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])

import gradio as gr
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import random, xml.etree.ElementTree as ET

# ── Đường dẫn (khớp chính xác với các cell trước) ───────────
PROJECT_ROOT       = Path("/kaggle/input/datasets/cnguynhong435345/llvip-model-weight/llvip_fusion_yolov8")
YOLO_RUNS_DIR      = PROJECT_ROOT / "runs"
EXPERIMENT_NAME    = "llvip_fusion_yolov8s"
BEST_MODEL_PATH    = YOLO_RUNS_DIR / EXPERIMENT_NAME / "weights" / "best.pt"
LAST_MODEL_PATH    = YOLO_RUNS_DIR / EXPERIMENT_NAME / "weights" / "last.pt"

# ── Auto-detect: quét thêm các vị trí khác phòng training lưu khác chỗ ──
def _find_pt_files() -> list:
    """Quét /kaggle/working tìm tất cả file .pt khả dụng."""
    import glob
    candidates = []
    # Các vị trí ưu tiên
    priority = [
        BEST_MODEL_PATH,
        LAST_MODEL_PATH,
        Path("/kaggle/working/yolov8s.pt"),
        Path("/kaggle/working/yolo26n.pt"),
    ]
    for p in priority:
        if p.exists():
            candidates.append(p)
    # Quét toàn bộ /kaggle/working tìm thêm .pt
    for pt in sorted(Path("/kaggle/working").rglob("*.pt")):
        if pt not in candidates:
            candidates.append(pt)
    return candidates

_ALL_PT_FILES = _find_pt_files()

# Build label→path map từ kết quả quét
_MODEL_MAP: dict = {}
for _p in _ALL_PT_FILES:
    # Nhãn hiển thị: ưu tiên rút gọn cho các path quen thuộc
    if _p == BEST_MODEL_PATH:
        _lbl = "best.pt (trained)"
    elif _p == LAST_MODEL_PATH:
        _lbl = "last.pt (trained)"
    elif _p.name == "yolov8s.pt":
        _lbl = "yolov8s.pt (pretrained)"
    elif _p.name == "yolo26n.pt":
        _lbl = "yolo26n.pt (pretrained)"
    else:
        # Rút gọn path: bỏ /kaggle/working/
        _lbl = str(_p).replace("/kaggle/working/", "")
    _MODEL_MAP[_lbl] = _p

print("=" * 55)
print("  CÁC MODEL TÌM THẤY TRONG /kaggle/working:")
print("=" * 55)
if _MODEL_MAP:
    for _lbl, _p in _MODEL_MAP.items():
        _sz = f"{_p.stat().st_size/1e6:.1f} MB"
        print(f"  ✅ {_lbl:<45} ({_sz})")
        print(f"     {_p}")
else:
    print("  ⚠️  Không tìm thấy file .pt nào!")
    print("  → Hãy chạy Cell 6 (training) trước.")
print("=" * 55)

FUSED_VAL          = PROJECT_ROOT / "images" / "val"
DATASET_LABELS_VAL = PROJECT_ROOT / "labels" / "val"

LLVIP_VISIBLE_TEST  = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/visible/test")
LLVIP_INFRARED_TEST = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/infrared/test")
LLVIP_ANNOTATIONS   = Path("/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/Annotations")

IMAGE_SIZE   = 640
FUSION_ALPHA = 0.5
FUSION_BETA  = 0.5

# ════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════

def resolve_model_path(choice: str) -> Path:
    # Tra cứu trực tiếp từ map được quét lúc khởi động
    if choice in _MODEL_MAP:
        return _MODEL_MAP[choice]
    # Fallback legacy labels
    if choice == "best.pt (trained)":
        return BEST_MODEL_PATH
    elif choice == "last.pt (trained)":
        return LAST_MODEL_PATH
    return Path("/kaggle/working/yolov8s.pt")

model_cache = {}
def get_model(choice: str) -> YOLO:
    path = resolve_model_path(choice)
    key  = str(path)
    if key not in model_cache:
        if not path.exists():
            raise FileNotFoundError(f"Không tìm thấy model: {path}")
        model_cache[key] = YOLO(str(path))
    return model_cache[key]

def fuse_pair(vis_bgr: np.ndarray, ir_bgr: np.ndarray,
              alpha: float = 0.5, beta: float = 0.5) -> np.ndarray:
    h, w = vis_bgr.shape[:2]
    if ir_bgr.shape[:2] != (h, w):
        ir_bgr = cv2.resize(ir_bgr, (w, h), interpolation=cv2.INTER_LINEAR)
    return cv2.addWeighted(vis_bgr, alpha, ir_bgr, beta, 0)

def draw_gt_boxes(img_bgr: np.ndarray, label_txt: Path,
                  color=(0, 220, 0), thickness=2) -> np.ndarray:
    """Vẽ ground-truth boxes từ file YOLO .txt lên ảnh."""
    out = img_bgr.copy()
    if not label_txt.exists():
        return out
    h, w = out.shape[:2]
    with open(label_txt) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, cx, cy, bw, bh = map(float, parts)
            x1 = int((cx - bw/2) * w);  y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w);  y2 = int((cy + bh/2) * h)
            cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
            cv2.putText(out, "GT", (x1, max(y1-4, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return out

def parse_gt_count_from_txt(label_txt: Path) -> int:
    if not label_txt.exists():
        return 0
    with open(label_txt) as f:
        return sum(1 for ln in f if len(ln.strip().split()) == 5)

def overlay_gt_pred(gt_bgr, pred_bgr) -> np.ndarray:
    """Ghép GT (trái) và Pred (phải) cạnh nhau."""
    h = max(gt_bgr.shape[0], pred_bgr.shape[0])
    def pad(img, h):
        dh = h - img.shape[0]
        return np.pad(img, ((0,dh),(0,0),(0,0)), mode='constant') if dh else img
    return np.concatenate([pad(gt_bgr, h), pad(pred_bgr, h)], axis=1)

def check_model_status() -> str:
    # Rescan mỗi lần bấm Refresh
    current = _find_pt_files()
    if not current:
        return "❌  Chưa tìm thấy file .pt nào trong /kaggle/working\n→ Hãy chạy Cell 6 (training) trước."
    lines = []
    for p in current:
        sz = f"({p.stat().st_size/1e6:.1f} MB)"
        short = str(p).replace("/kaggle/working/", "")
        lines.append(f"✅  {short:<50} {sz}\n   📂 {p}")
    return "\n\n".join(lines)

# ════════════════════════════════════════════════════════════
# TAB 1 – Upload ảnh bất kỳ
# ════════════════════════════════════════════════════════════

def predict_uploaded(image_np, model_choice, conf, iou, show_labels, show_conf):
    try:
        model = get_model(model_choice)
    except FileNotFoundError as e:
        return None, str(e)
    bgr     = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
    results = model.predict(source=bgr, conf=conf, iou=iou,
                            imgsz=IMAGE_SIZE, device="cpu", verbose=False)
    r       = results[0]
    ann     = cv2.cvtColor(r.plot(labels=show_labels, conf=show_conf), cv2.COLOR_BGR2RGB)
    n       = len(r.boxes)
    confs   = r.boxes.conf.cpu().numpy().tolist() if n else []
    info    = (f"✅ Phát hiện: {n} người đi bộ\n"
               f"📊 Conf TB: {np.mean(confs):.3f}\n"
               f"🔧 Model: {model_choice}\n"
               f"⚙️  conf={conf:.2f} | iou={iou:.2f}")
    return ann, info

# ════════════════════════════════════════════════════════════
# TAB 2 – Fusion 2 ảnh thủ công
# ════════════════════════════════════════════════════════════

def predict_fused_pair(vis_np, ir_np, model_choice, conf, iou,
                        alpha, show_labels, show_conf):
    if vis_np is None or ir_np is None:
        return None, None, "⚠️ Vui lòng upload cả ảnh visible VÀ infrared."
    try:
        model = get_model(model_choice)
    except FileNotFoundError as e:
        return None, None, str(e)
    vis_bgr   = cv2.cvtColor(vis_np, cv2.COLOR_RGB2BGR)
    ir_bgr    = cv2.cvtColor(cv2.cvtColor(ir_np, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2BGR)
    beta      = round(1.0 - alpha, 2)
    fused_bgr = fuse_pair(vis_bgr, ir_bgr, alpha, beta)
    results   = model.predict(source=fused_bgr, conf=conf, iou=iou,
                              imgsz=IMAGE_SIZE, device="cpu", verbose=False)
    r         = results[0]
    n         = len(r.boxes)
    confs     = r.boxes.conf.cpu().numpy().tolist() if n else []
    info      = (f"✅ Phát hiện: {n} người đi bộ\n"
                 f"📊 Conf TB: {np.mean(confs):.3f}\n"
                 f"🔀 α(vis)={alpha:.2f} | β(ir)={beta:.2f}\n"
                 f"🔧 Model: {model_choice}")
    return (cv2.cvtColor(fused_bgr, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(r.plot(labels=show_labels, conf=show_conf), cv2.COLOR_BGR2RGB),
            info)

# ════════════════════════════════════════════════════════════
# TAB 3 – Mẫu ngẫu nhiên tập val (fused)
# ════════════════════════════════════════════════════════════

def sample_val_image(model_choice, conf, iou, n_samples, show_labels, show_conf):
    try:
        model = get_model(model_choice)
    except FileNotFoundError as e:
        return [], str(e)
    src_dir = FUSED_VAL if (FUSED_VAL.exists() and
                            list(FUSED_VAL.glob("*.jpg"))) else LLVIP_VISIBLE_TEST
    all_imgs = sorted(src_dir.glob("*.jpg")) + sorted(src_dir.glob("*.png"))
    if not all_imgs:
        return [], f"⚠️ Không tìm thấy ảnh trong: {src_dir}"
    samples   = random.sample(all_imgs, min(int(n_samples), len(all_imgs)))
    gallery   = []
    total_det = 0
    for p in samples:
        bgr = cv2.imread(str(p))
        r   = model.predict(source=bgr, conf=conf, iou=iou,
                            imgsz=IMAGE_SIZE, device="cpu", verbose=False)[0]
        n   = len(r.boxes); total_det += n
        gallery.append((cv2.cvtColor(r.plot(labels=show_labels, conf=show_conf),
                                     cv2.COLOR_BGR2RGB),
                        f"{p.name} | {n} detected"))
    info = (f"🖼️  {len(samples)} ảnh | Tổng phát hiện: {total_det}\n"
            f"📁 Nguồn: {src_dir}\n"
            f"🔧 {model_choice} | conf={conf:.2f}")
    return gallery, info

# ════════════════════════════════════════════════════════════
# TAB 4 – Cặp ảnh RAW test (Visible + Infrared + Fused + Pred)
# ════════════════════════════════════════════════════════════

def sample_raw_test_pair(model_choice, conf, iou, alpha,
                          show_labels, show_conf, show_gt):
    """
    Lấy ngẫu nhiên 1 cặp ảnh thực từ tập test LLVIP:
      visible/test  +  infrared/test  +  Annotations (GT)
    Trả về 5 ảnh: Visible | Infrared | Fused | GT | Prediction
    """
    try:
        model = get_model(model_choice)
    except FileNotFoundError as e:
        return None, None, None, None, None, str(e)

    # Tìm ảnh visible test
    vis_dir = LLVIP_VISIBLE_TEST
    if not vis_dir.exists() or not list(vis_dir.glob("*.jpg")):
        return None, None, None, None, None, \
               f"❌ Không tìm thấy visible/test: {vis_dir}"

    all_vis = sorted(vis_dir.glob("*.jpg")) + sorted(vis_dir.glob("*.png"))
    vis_path = random.choice(all_vis)
    stem     = vis_path.stem

    # Tìm ảnh infrared cùng tên
    ir_dir    = LLVIP_INFRARED_TEST
    ir_path   = ir_dir / vis_path.name
    if not ir_path.exists():
        cands = list(ir_dir.glob(f"{stem}.*"))
        ir_path = cands[0] if cands else None

    # Đọc visible
    vis_bgr = cv2.imread(str(vis_path))
    if vis_bgr is None:
        return None, None, None, None, None, f"❌ Không đọc được: {vis_path}"

    # Đọc infrared (nếu có)
    if ir_path and ir_path.exists():
        ir_gray = cv2.imread(str(ir_path), cv2.IMREAD_GRAYSCALE)
        ir_bgr  = cv2.cvtColor(ir_gray, cv2.COLOR_GRAY2BGR)
    else:
        ir_bgr = np.zeros_like(vis_bgr)

    beta      = round(1.0 - alpha, 2)
    fused_bgr = fuse_pair(vis_bgr, ir_bgr, alpha, beta)

    # Ground-truth từ Annotations XML
    gt_n   = 0
    gt_bgr = fused_bgr.copy()
    xml_path = LLVIP_ANNOTATIONS / f"{stem}.xml"
    if xml_path.exists():
        try:
            tree = ET.parse(str(xml_path))
            root = tree.getroot()
            h_img, w_img = gt_bgr.shape[:2]
            for obj in root.findall("object"):
                cls = obj.find("name").text.strip().lower()
                if cls not in ("person", "pedestrian"):
                    continue
                bb = obj.find("bndbox")
                x1 = int(float(bb.find("xmin").text))
                y1 = int(float(bb.find("ymin").text))
                x2 = int(float(bb.find("xmax").text))
                y2 = int(float(bb.find("ymax").text))
                cv2.rectangle(gt_bgr, (x1,y1), (x2,y2), (0,220,0), 2)
                cv2.putText(gt_bgr, "GT", (x1, max(y1-5,0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,220,0), 1)
                gt_n += 1
        except Exception:
            pass
    # Fallback: YOLO label txt
    elif (DATASET_LABELS_VAL / f"{stem}.txt").exists():
        lbl_path = DATASET_LABELS_VAL / f"{stem}.txt"
        gt_bgr   = draw_gt_boxes(fused_bgr, lbl_path)
        gt_n     = parse_gt_count_from_txt(lbl_path)

    # Prediction
    results   = model.predict(source=fused_bgr, conf=conf, iou=iou,
                              imgsz=IMAGE_SIZE, device="cpu", verbose=False)
    r         = results[0]
    pred_n    = len(r.boxes)
    confs_    = r.boxes.conf.cpu().numpy().tolist() if pred_n else []
    pred_bgr  = r.plot(labels=show_labels, conf=show_conf)

    # Tạo header label cho mỗi ảnh
    def add_header(img_bgr, text, color=(255,255,255)):
        header = np.full((30, img_bgr.shape[1], 3), 40, dtype=np.uint8)
        cv2.putText(header, text, (6, 21),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1, cv2.LINE_AA)
        return np.vstack([header, img_bgr])

    vis_h    = add_header(vis_bgr,   "VISIBLE (RGB)",       (150,220,255))
    ir_h     = add_header(ir_bgr,    "INFRARED (Thermal)",  (100,255,200))
    fused_h  = add_header(fused_bgr, f"FUSED  a={alpha:.2f}/b={beta:.2f}", (255,230,100))
    gt_h     = add_header(gt_bgr,    f"GROUND TRUTH  [{gt_n} ped]",  (80,220,80))
    pred_h   = add_header(pred_bgr,  f"PREDICTION    [{pred_n} ped]", (80,180,255))

    to_rgb = lambda x: cv2.cvtColor(x, cv2.COLOR_BGR2RGB)

    # Thống kê
    missed   = max(gt_n - pred_n, 0)
    extra    = max(pred_n - gt_n, 0)
    avg_conf = np.mean(confs_) if confs_ else 0.0
    ir_status = f"{ir_path}" if ir_path and ir_path.exists() else "⚠️ Không có infrared"
    gt_status = "XML Annotation" if xml_path.exists() else "YOLO label.txt"
    info = (
        f"🖼️  File     : {vis_path.name}\n"
        f"🌡️  Infrared : {ir_status}\n"
        f"📌 GT source : {gt_status}\n"
        f"──────────────────────────────────\n"
        f"👤 GT pedestrians   : {gt_n}\n"
        f"🎯 Predicted         : {pred_n}\n"
        f"📊 Avg confidence    : {avg_conf:.3f}\n"
        f"❌ Missed (FN)       : {missed}\n"
        f"➕ Extra (FP)        : {extra}\n"
        f"🔀 Fusion α={alpha:.2f} β={beta:.2f}\n"
        f"🔧 Model: {model_choice}"
    )

    out_vis   = to_rgb(vis_h)
    out_ir    = to_rgb(ir_h)
    out_fused = to_rgb(fused_h)
    out_gt    = to_rgb(gt_h)    if show_gt else None
    out_pred  = to_rgb(pred_h)

    return out_vis, out_ir, out_fused, out_gt, out_pred, info


def sample_raw_multi(model_choice, conf, iou, alpha,
                      n_samples, show_labels, show_conf):
    """
    Lấy N cặp ngẫu nhiên từ raw test, trả về gallery
    mỗi item = ảnh so sánh GT | Prediction ghép ngang.
    """
    try:
        model = get_model(model_choice)
    except FileNotFoundError as e:
        return [], str(e)

    vis_dir  = LLVIP_VISIBLE_TEST
    ir_dir   = LLVIP_INFRARED_TEST
    if not vis_dir.exists():
        return [], f"❌ Không tìm thấy: {vis_dir}"

    all_vis  = sorted(vis_dir.glob("*.jpg")) + sorted(vis_dir.glob("*.png"))
    if not all_vis:
        return [], "⚠️ Không có ảnh trong visible/test"

    samples   = random.sample(all_vis, min(int(n_samples), len(all_vis)))
    gallery   = []
    total_gt  = 0
    total_pred= 0
    beta      = round(1.0 - alpha, 2)

    for vis_path in samples:
        stem    = vis_path.stem
        vis_bgr = cv2.imread(str(vis_path))
        if vis_bgr is None:
            continue

        # Infrared
        ir_path = ir_dir / vis_path.name
        if not ir_path.exists():
            cands   = list(ir_dir.glob(f"{stem}.*"))
            ir_path = cands[0] if cands else None
        if ir_path and ir_path.exists():
            ir_bgr = cv2.cvtColor(cv2.imread(str(ir_path),
                                             cv2.IMREAD_GRAYSCALE),
                                   cv2.COLOR_GRAY2BGR)
        else:
            ir_bgr = np.zeros_like(vis_bgr)

        fused_bgr = fuse_pair(vis_bgr, ir_bgr, alpha, beta)

        # GT từ XML
        gt_bgr  = fused_bgr.copy()
        gt_n    = 0
        xml_p   = LLVIP_ANNOTATIONS / f"{stem}.xml"
        if xml_p.exists():
            try:
                root = ET.parse(str(xml_p)).getroot()
                h_, w_ = gt_bgr.shape[:2]
                for obj in root.findall("object"):
                    if obj.find("name").text.strip().lower() not in ("person","pedestrian"):
                        continue
                    bb = obj.find("bndbox")
                    x1=int(float(bb.find("xmin").text)); y1=int(float(bb.find("ymin").text))
                    x2=int(float(bb.find("xmax").text)); y2=int(float(bb.find("ymax").text))
                    cv2.rectangle(gt_bgr,(x1,y1),(x2,y2),(0,220,0),2)
                    gt_n += 1
            except Exception:
                pass

        # Prediction
        r      = model.predict(source=fused_bgr, conf=conf, iou=iou,
                               imgsz=IMAGE_SIZE, device="cpu", verbose=False)[0]
        pred_n = len(r.boxes)
        pred_bgr = r.plot(labels=show_labels, conf=show_conf)

        total_gt   += gt_n
        total_pred += pred_n

        # Dán nhãn lên ảnh rồi ghép ngang
        def stamp(img, txt, c):
            i = img.copy()
            cv2.rectangle(i, (0,0), (i.shape[1], 26), (20,20,20), -1)
            cv2.putText(i, txt, (5,19), cv2.FONT_HERSHEY_SIMPLEX, 0.55, c, 1, cv2.LINE_AA)
            return i

        gt_stamp   = stamp(gt_bgr,   f"GT [{gt_n}]",   (80,220,80))
        pred_stamp = stamp(pred_bgr, f"PRED [{pred_n}]", (80,180,255))
        side_by_side = overlay_gt_pred(gt_stamp, pred_stamp)
        gallery.append(
            (cv2.cvtColor(side_by_side, cv2.COLOR_BGR2RGB),
             f"{stem} | GT={gt_n} | Pred={pred_n}")
        )

    info = (
        f"📦 {len(gallery)} cặp ảnh từ raw test dataset\n"
        f"👤 Tổng GT:   {total_gt}\n"
        f"🎯 Tổng Pred: {total_pred}\n"
        f"🔀 Fusion α={alpha:.2f} β={beta:.2f} | conf={conf:.2f}\n"
        f"📁 {vis_dir}"
    )
    return gallery, info

# ════════════════════════════════════════════════════════════
# BUILD GRADIO UI
# ════════════════════════════════════════════════════════════

# Dùng danh sách đã quét thực tế
available_models = list(_MODEL_MAP.keys()) or ["best.pt (trained)"]

with gr.Blocks(
    title="LLVIP Pedestrian Detection",
    theme=gr.themes.Soft(primary_hue="blue")
) as demo:

    gr.Markdown("""
    # 🔦 Low-Light Pedestrian Detection — LLVIP + YOLOv8s
    **Dataset:** LLVIP (Visible + Infrared paired) &nbsp;|&nbsp;
    **Model:** YOLOv8s fine-tuned trên fused images (α=0.5 visible + β=0.5 infrared)
    """)

    with gr.Accordion("📋 Trạng thái Model", open=False):
        status_box = gr.Textbox(value=check_model_status(), lines=9,
                                label="Model files", interactive=False)
        gr.Button("🔄 Refresh").click(fn=check_model_status, outputs=status_box)

    # ── Tham số chung ────────────────────────────────────────
    with gr.Row():
        g_model = gr.Dropdown(available_models, value=available_models[0],
                              label="🔧 Model", scale=2)
        g_conf  = gr.Slider(0.05, 0.95, value=0.25, step=0.05,
                            label="Confidence Threshold", scale=2)
        g_iou   = gr.Slider(0.05, 0.95, value=0.45, step=0.05,
                            label="IoU Threshold", scale=2)
    with gr.Row():
        g_lbl  = gr.Checkbox(True,  label="Hiển thị nhãn class")
        g_conf_show = gr.Checkbox(True, label="Hiển thị confidence score")

    gr.Markdown("---")

    with gr.Tabs():

        # ── Tab 1 ────────────────────────────────────────────
        with gr.Tab("📁 Upload ảnh"):
            gr.Markdown("Upload ảnh bất kỳ để detect người đi bộ.")
            with gr.Row():
                t1_in  = gr.Image(type="numpy", label="📷 Input", height=380)
                t1_out = gr.Image(type="numpy", label="🎯 Result", height=380)
            t1_info = gr.Textbox(lines=5, label="📊 Thông tin", interactive=False)
            gr.Button("🚀 Detect", variant="primary").click(
                fn=predict_uploaded,
                inputs=[t1_in, g_model, g_conf, g_iou, g_lbl, g_conf_show],
                outputs=[t1_out, t1_info]
            )

        # ── Tab 2 ────────────────────────────────────────────
        with gr.Tab("🔀 Fusion thủ công"):
            gr.Markdown("Upload riêng ảnh Visible + Infrared — tự động fusion rồi detect.")
            with gr.Row():
                t2_vis = gr.Image(type="numpy", label="🌈 Visible",  height=280)
                t2_ir  = gr.Image(type="numpy", label="🌡️ Infrared", height=280)
            t2_alpha = gr.Slider(0.1, 0.9, 0.5, step=0.05,
                                  label="Alpha (Visible ratio) — Beta = 1 - Alpha")
            with gr.Row():
                t2_fused  = gr.Image(type="numpy", label="🔀 Fused",     height=340)
                t2_result = gr.Image(type="numpy", label="🎯 Detection", height=340)
            t2_info = gr.Textbox(lines=5, label="📊 Thông tin", interactive=False)
            gr.Button("🚀 Fusion & Detect", variant="primary").click(
                fn=predict_fused_pair,
                inputs=[t2_vis, t2_ir, g_model, g_conf, g_iou,
                        t2_alpha, g_lbl, g_conf_show],
                outputs=[t2_fused, t2_result, t2_info]
            )

        # ── Tab 3 ────────────────────────────────────────────
        with gr.Tab("🎲 Mẫu Val Set (fused)"):
            gr.Markdown("Lấy ngẫu nhiên từ **fused validation set** đã tạo sẵn.")
            t3_n = gr.Slider(1, 12, 4, step=1, label="Số ảnh mẫu")
            gr.Button("🎲 Sample & Detect", variant="primary").click(
                fn=sample_val_image,
                inputs=[g_model, g_conf, g_iou, t3_n, g_lbl, g_conf_show],
                outputs=[
                    gr.Gallery(label="🖼️ Results", columns=2, height=560,
                               object_fit="contain"),
                    gr.Textbox(lines=4, label="📊 Thống kê", interactive=False)
                ]
            )

        # ── Tab 4 (MỚI) ──────────────────────────────────────
        with gr.Tab("🧪 Raw Test Pair (Vis+IR → GT vs Pred)"):
            gr.Markdown("""
            ### Lấy ngẫu nhiên cặp ảnh gốc từ tập **test** LLVIP
            Hiển thị đầy đủ: **Visible** | **Infrared** | **Ảnh Fused** | **Ground Truth** | **Prediction**  
            GT được đọc trực tiếp từ file **Annotations XML** gốc của LLVIP.
            """)

            with gr.Row():
                t4_alpha  = gr.Slider(0.1, 0.9, 0.5, step=0.05,
                                      label="Fusion Alpha (Visible)", scale=3)
                t4_show_gt = gr.Checkbox(True, label="Hiển thị Ground Truth", scale=1)

            # ── Single random pair ────────────────────────────
            with gr.Accordion("🔍 Chi tiết 1 cặp ảnh ngẫu nhiên", open=True):
                gr.Markdown("Mỗi lần nhấn lấy **1 cặp ảnh ngẫu nhiên** khác nhau, hiển thị từng bước pipeline.")
                with gr.Row():
                    t4_vis   = gr.Image(type="numpy", label="🌈 Visible (RGB)",    height=260)
                    t4_ir    = gr.Image(type="numpy", label="🌡️ Infrared (Thermal)", height=260)
                    t4_fused = gr.Image(type="numpy", label="🔀 Fused Image",      height=260)
                with gr.Row():
                    t4_gt    = gr.Image(type="numpy", label="✅ Ground Truth (XML)", height=260)
                    t4_pred  = gr.Image(type="numpy", label="🎯 Model Prediction",  height=260)
                t4_info  = gr.Textbox(lines=10, label="📊 Phân tích chi tiết",
                                      interactive=False)
                gr.Button("🎲 Lấy cặp ảnh ngẫu nhiên", variant="primary").click(
                    fn=sample_raw_test_pair,
                    inputs=[g_model, g_conf, g_iou, t4_alpha,
                            g_lbl, g_conf_show, t4_show_gt],
                    outputs=[t4_vis, t4_ir, t4_fused, t4_gt, t4_pred, t4_info]
                )

            gr.Markdown("---")

            # ── Multi pair gallery ────────────────────────────
            with gr.Accordion("📷 Gallery nhiều cặp (GT | Pred ghép ngang)", open=True):
                gr.Markdown("Lấy **N cặp ảnh** và hiển thị **Ground Truth vs Prediction** đặt cạnh nhau.")
                t4_n = gr.Slider(2, 16, 6, step=1, label="Số cặp ảnh")
                gr.Button("🎲 Sample nhiều cặp", variant="primary").click(
                    fn=sample_raw_multi,
                    inputs=[g_model, g_conf, g_iou, t4_alpha,
                            t4_n, g_lbl, g_conf_show],
                    outputs=[
                        gr.Gallery(label="🖼️ GT | Prediction (ghép ngang)",
                                   columns=2, height=700, object_fit="contain"),
                        gr.Textbox(lines=6, label="📊 Thống kê tổng hợp", interactive=False)
                    ]
                )

    gr.Markdown("""
    ---
    **Đường dẫn model:** `best.pt` → `/kaggle/working/llvip_fusion_yolov8/runs/llvip_fusion_yolov8s/weights/best.pt`  
    **Dataset test:**  `/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/visible/test` + `infrared/test`  
    **Annotations:** `/kaggle/input/datasets/cnguynhong435345/lllvipp/LLVIP/Annotations`
    """)


# ── Đóng instance Gradio cũ (tránh lỗi port in use) ────
try:
    demo.close()
except Exception:
    pass
try:
    import gradio as _gr
    _gr.close_all()
except Exception:
    pass

demo.launch(
    share=True,
    debug=False,
    show_error=True,
    server_port=None,   # tự tìm port trống 7860, 7861...
)


## Cell OFFLINE — Trích xuất toàn bộ chỉ số đánh giá (không chạy lại mô hình)
> Chỉ cần có thư mục  từ lần training trước.
> Đọc trực tiếp từ  và các file ảnh Ultralytics đã sinh sẵn.

In [ ]:
# ============================================================
# CELL OFFLINE — Trích xuất toàn bộ chỉ số đánh giá
# KHÔNG chạy lại mô hình — chỉ đọc từ runs/ đã lưu
# ============================================================

import csv
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image  as mpimg
import numpy  as np

# ── Đường dẫn CỐ ĐỊNH theo cấu trúc dataset Kaggle ──────────
RUN_DIR    = Path("/kaggle/input/datasets/cnguynhong435345/llvip-model-weight/llvip_fusion_yolov8/runs/llvip_fusion_yolov8s")
CSV_PATH   = RUN_DIR / "results.csv"

# OUTPUT luôn ghi vào /kaggle/working — có quyền ghi
OUTPUT_DIR = Path("/kaggle/working/offline_metrics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEP = "=" * 60

if not RUN_DIR.exists():
    raise FileNotFoundError(f"\n  [ERROR] Không tìm thấy: {RUN_DIR}")

print(f"  [OK] Đọc từ  : {RUN_DIR}")
print(f"  [OK] Lưu vào : {OUTPUT_DIR}")

# ============================================================
# PHẦN 1 — ĐỌC results.csv & TÍNH TOÁN TOÀN BỘ CHỈ SỐ
# ============================================================

def load_results_csv(csv_path: Path) -> list:
    if not csv_path.exists():
        raise FileNotFoundError(f"Không tìm thấy: {csv_path}")
    rows = []
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            clean = {}
            for k, v in row.items():
                k, v = k.strip(), v.strip()
                if v:
                    try:    clean[k] = float(v)
                    except: clean[k] = v
            if clean:
                rows.append(clean)
    return rows


def extract_best_epoch_metrics(rows: list) -> dict:
    best_row  = max(rows, key=lambda r: r.get("metrics/mAP50(B)", 0))
    precision = best_row.get("metrics/precision(B)", float("nan"))
    recall    = best_row.get("metrics/recall(B)",    float("nan"))
    map50     = best_row.get("metrics/mAP50(B)",     float("nan"))
    map5095   = best_row.get("metrics/mAP50-95(B)",  float("nan"))
    f1        = 2*precision*recall/(precision+recall+1e-9)
    return {
        "best_epoch"    : int(best_row.get("epoch", -1)) + 1,
        "Precision"     : precision,
        "Recall"        : recall,
        "mAP@50"        : map50,
        "mAP@50-95"     : map5095,
        "F1-Score"      : f1,
        "train/box_loss": best_row.get("train/box_loss", float("nan")),
        "train/cls_loss": best_row.get("train/cls_loss", float("nan")),
        "train/dfl_loss": best_row.get("train/dfl_loss", float("nan")),
        "val/box_loss"  : best_row.get("val/box_loss",   float("nan")),
        "val/cls_loss"  : best_row.get("val/cls_loss",   float("nan")),
    }


def print_offline_metrics_table(m: dict) -> None:
    print(f"\n{SEP}")
    print("    OFFLINE EVALUATION RESULTS (từ results.csv)")
    print(f"    Best checkpoint: Epoch {m['best_epoch']}")
    print(SEP)
    for key in ["Precision", "Recall", "mAP@50", "mAP@50-95", "F1-Score"]:
        val  = m[key]
        bar  = "█" * int(val*30) + "░" * (30 - int(val*30))
        flag = "✓" if val >= 0.5 else "△"
        print(f"   {flag}  {key:<18}: {val:.4f}  ({val*100:.2f}%)  |{bar}|")
    print(SEP)
    for key, label in [
        ("train/box_loss","Train Box Loss"), ("train/cls_loss","Train Cls Loss"),
        ("train/dfl_loss","Train DFL Loss"), ("val/box_loss",  "Val   Box Loss"),
        ("val/cls_loss",  "Val   Cls Loss"),
    ]:
        print(f"      {label:<20}: {m.get(key, float('nan')):.4f}")
    print(SEP)


print("Đang đọc results.csv ...")
training_rows = load_results_csv(CSV_PATH)
best_metrics  = extract_best_epoch_metrics(training_rows)
print_offline_metrics_table(best_metrics)

offline_metric_values = {
    k: best_metrics[k]
    for k in ["Precision", "Recall", "mAP@50", "mAP@50-95", "F1-Score"]
}

# ============================================================
# PHẦN 2 — VẼ TOÀN BỘ ĐƯỜNG CONG TỪ results.csv
# ============================================================

def plot_all_curves_offline(rows: list, output_path: Path) -> None:
    epochs         = [r.get("epoch", i)+1         for i,r in enumerate(rows)]
    train_box_loss = [r.get("train/box_loss",   0) for r in rows]
    train_cls_loss = [r.get("train/cls_loss",   0) for r in rows]
    train_dfl_loss = [r.get("train/dfl_loss",   0) for r in rows]
    val_box_loss   = [r.get("val/box_loss",     0) for r in rows]
    val_cls_loss   = [r.get("val/cls_loss",     0) for r in rows]
    map50_curve    = [r.get("metrics/mAP50(B)", 0) for r in rows]
    map5095_curve  = [r.get("metrics/mAP50-95(B)", 0) for r in rows]
    prec_curve     = [r.get("metrics/precision(B)", 0) for r in rows]
    rec_curve      = [r.get("metrics/recall(B)",    0) for r in rows]
    f1_curve       = [2*p*r/(p+r+1e-9) for p,r in zip(prec_curve, rec_curve)]
    best_ep        = best_metrics["best_epoch"]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("YOLOv8s Training & Validation Curves — LLVIP Fused Dataset",
                 fontsize=14, fontweight="bold", y=1.01)

    def _ax(ax, x, series, title, ylabel):
        for y_data, color, label, ls in series:
            ax.plot(x, y_data, color=color, linewidth=2, label=label, linestyle=ls)
        ax.axvline(x=best_ep, color="gray", linestyle=":", linewidth=1.2,
                   label=f"Best (ep {best_ep})")
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(fontsize=9)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        ax.grid(axis="y", alpha=0.3)

    _ax(axes[0][0], epochs,
        [(train_box_loss,"#E15759","Train","-"),(val_box_loss,"#E15759","Val","--")],
        "Box Loss","Loss")
    _ax(axes[0][1], epochs,
        [(train_cls_loss,"#F28E2B","Train","-"),(val_cls_loss,"#F28E2B","Val","--")],
        "Cls Loss","Loss")
    _ax(axes[0][2], epochs,
        [(train_dfl_loss,"#76B7B2","Train","-")],
        "DFL Loss","Loss")
    _ax(axes[1][0], epochs,
        [(map50_curve,"#4E79A7","mAP@50","-"),(map5095_curve,"#59A14F","mAP@50-95","--")],
        "mAP Curves","mAP")
    _ax(axes[1][1], epochs,
        [(prec_curve,"#B07AA1","Precision","-"),(rec_curve,"#FF9DA7","Recall","--")],
        "Precision & Recall","Score")
    _ax(axes[1][2], epochs,
        [(f1_curve,"#EDC948","F1-Score","-")],
        "F1-Score Curve","F1")

    plt.tight_layout()
    plt.savefig(str(output_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  [SAVED] → {output_path}")


print("\nĐang vẽ đường cong huấn luyện ...")
plot_all_curves_offline(training_rows, OUTPUT_DIR / "offline_training_curves.png")

# ============================================================
# PHẦN 3 — BIỂU ĐỒ CỘT TỔNG HỢP CHỈ SỐ CUỐI
# ============================================================

def plot_offline_bar_chart(metric_values: dict, output_path: Path) -> None:
    names  = list(metric_values.keys())
    scores = list(metric_values.values())
    colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, scores, color=colors, edgecolor="white", linewidth=1.2, width=0.55)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.012,
                f"{score:.4f}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"Detection Performance — YOLOv8s LLVIP (Best Epoch {best_metrics['best_epoch']})",
                 fontsize=13, fontweight="bold", pad=15)
    ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="0.5 threshold")
    ax.axhline(y=1.0, color="black", linestyle="-", linewidth=0.8, alpha=0.3)
    ax.legend(fontsize=10)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(str(output_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  [SAVED] → {output_path}")


print("\nĐang vẽ biểu đồ cột chỉ số ...")
plot_offline_bar_chart(offline_metric_values, OUTPUT_DIR / "offline_metrics_bar.png")

# ============================================================
# PHẦN 4 — HIỂN THỊ CÁC ẢNH ULTRALYTICS TỰ SINH
# ============================================================

ULTRALYTICS_PLOTS = {
    "PR Curve"                      : RUN_DIR / "BoxPR_curve.png",
    "F1 Curve"                      : RUN_DIR / "BoxF1_curve.png",
    "Precision Curve"               : RUN_DIR / "BoxP_curve.png",
    "Recall Curve"                  : RUN_DIR / "BoxR_curve.png",
    "Confusion Matrix"              : RUN_DIR / "confusion_matrix.png",
    "Confusion Matrix (Normalized)" : RUN_DIR / "confusion_matrix_normalized.png",
}

available_plots = {t: p for t, p in ULTRALYTICS_PLOTS.items() if p.exists()}

if available_plots:
    n_cols = min(len(available_plots), 3)
    n_rows = (len(available_plots) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7*n_cols, 6*n_rows))
    axes = np.array(axes).flatten()

    for idx, (title, img_path) in enumerate(available_plots.items()):
        axes[idx].imshow(mpimg.imread(str(img_path)))
        axes[idx].set_title(title, fontsize=12, fontweight="bold", pad=8)
        axes[idx].axis("off")
    for idx in range(len(available_plots), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle("Ultralytics Auto-Generated Evaluation Plots",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    combined_path = OUTPUT_DIR / "offline_ultralytics_plots.png"
    plt.savefig(str(combined_path), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  [SAVED] → {combined_path}")

    print("\n  [COPY] Copy sang /kaggle/working/offline_metrics/:")
    for title, img_path in available_plots.items():
        dest = OUTPUT_DIR / img_path.name
        shutil.copy2(str(img_path), str(dest))
        print(f"    • {img_path.name}")
else:
    print(f"\n  [WARNING] Không tìm thấy file ảnh trong: {RUN_DIR}")

# ============================================================
# PHẦN 5 — TÓM TẮT FILE ĐẦU RA
# ============================================================
print(f"\n{SEP}")
print("    TÓM TẮT ĐẦU RA  →  /kaggle/working/offline_metrics/")
print(SEP)
for f in sorted(OUTPUT_DIR.glob("*.png")):
    print(f"    ✓  {f.name:<45} ({f.stat().st_size/1024:>7.1f} KB)")
print(SEP)
print(f"  offline_metric_values = {offline_metric_values}")
print(SEP)